# Guia Completo de Dashboards com Streamlit
### Apps Prontos, Explicados Módulo a Módulo

---
> **Diferente dos tutoriais anteriores**, o Streamlit não roda dentro do Jupyter: cada app é um arquivo **`.py`** executado no terminal com:
>
> `streamlit run app_01_primeiro_app.py`
>
> Os apps prontos estão nesta mesma pasta. Na pasta do arquivo, rode o comando acima e o navegador abrirá a página.

## O que é o Streamlit

Framework **Python** que transforma scripts em **aplicações web de dados**. Ideia central: você programa como um script normal (import pandas, carrega arquivo, monta um gráfico) e o Streamlit cuida de toda a parte web.

Dois comportamentos-chave:

- **Reexecução completa:** cada interação do usuário reroda o script de cima a baixo.
- **Estado:** `st.session_state` guarda valores entre as reexecuções.

Instalação (uma única vez): `pip install streamlit`

## Os 9 apps deste tutorial

| App | Arquivo | Tema |
|-----|---------|------|
| 1 | `app_01_primeiro_app.py` | Títulos, `st.write`, primeira tabela |
| 2 | `app_02_dados_tabela.py` | `st.dataframe`, `st.table`, `describe` |
| 3 | `app_03_graficos.py` | `st.line_chart`, `st.bar_chart`, `st.area_chart` |
| 4 | `app_04_widgets.py` | `selectbox`, `multiselect`, `slider` |
| 5 | `app_05_sidebar_layout.py` | Sidebar, colunas, expander, abas |
| 6 | `app_06_metricas_form.py` | `st.metric`, formulário, download, sessão |
| 7 | `app_07_pyplot_matplotlib.py` | Matplotlib dentro do app (`st.pyplot`) |
| 8 | `app_08_cache.py` | `st.cache_data` para acelerar recargas |
| 9 | `app_09_dashboard.py` | Dashboard completo de vendas (projeto final) |

## Configuração do Ambiente

As células de código deste notebook fazem a **preparação de dados** (Python/pandas) que os apps reutilizam — e verificam que todos os `.py` estão corretos. As instruções de Streamlit em si ficam nos arquivos.

In [ ]:
import pandas as pd

# Carrega o mesmo dado usado nos apps e cria a coluna de mes
df = pd.read_csv('vendas.csv', sep=';')
df['mes'] = df['data_hora'].str[:7]

print('Dados carregados:', df.shape)
print('Periodo:', df['mes'].min(), 'a', df['mes'].max())

---
# MÓDULO 1 — Nível Básico
## Primeiros comandos e gráficos nativos

## 1.1 Primeiro App: `app_01_primeiro_app.py`

Peças para estruturar a página (como cabeçalhos de documento):

```python
import streamlit as st

st.title('Meu Primeiro App')     # titulo principal
st.header('Tema: Analise')       # secao
st.subheader('Subtitulo')        # subsecao
st.write(texto_ou_dado)          # elemento universal
st.dataframe(df)                 # tabela interativa
```

`st.write` aceita texto com marcação simples (`**negrito**`, `_italico_`), DataFrames, dicionários, figuras etc.

**Rode:** `streamlit run app_01_primeiro_app.py`

## 1.2 Exibindo Dados: `app_02_dados_tabela.py`

- `st.dataframe(df)` — tabela **interativa**: rolagem, ordenação por coluna, tamanho ajustável. Use sempre que houver muitas linhas.
- `st.table(df)` — tabela **estática**, sem interação. Boa para poucas linhas fixas.
- Dá para passar qualquer objeto pandas, inclusive o resultado de `df.describe()`.
- O parâmetro `height` limita a altura visível, deixando a rolagem interna.

## 1.3 Gráficos Nativos: `app_03_graficos.py`

Três comandos geram gráficos direto do DataFrame, sem biblioteca extra:

```python
st.line_chart(data)   # linha
st.bar_chart(data)    # barras
st.area_chart(data)   # area
```

Bastante atenção à **preparação dos dados** (groupby/agregação), que é o que garante um gráfico legível. Veja o mesmo cálculo abaixo:

In [ ]:
# Mesmo calculo do app_03: pedidos e receita por mes (agrupar ANTES de plotar)
mensal = df.groupby('mes').agg(
    pedidos=('cliente_id', 'count'),
    receita=('valor_venda', 'sum'),
).sort_index()

mensal

In [ ]:
# No app, bastaria: st.line_chart(mensal['pedidos'])
print(mensal['pedidos'].to_string())

# MÓDULO 2 — Nível Intermediário
## Interatividade, layout, métricas e formulários

## 2.1 Widgets: `app_04_widgets.py`

Widgets devolvem valores que usamos para **filtrar** os dados:

```python
estado = st.sidebar.selectbox('Estado', options=['Todos'] + lista)
cats   = st.sidebar.multiselect('Categorias', options=lista, default=lista)
lo, hi = st.sidebar.slider('Faixa', min_value=0, max_value=5000, value=(0, 5000))
```

Regra de ouro: **filtre o DataFrame com o valor dos widgets**, nunca o contrário. Alterar um widget reroda o script e a tela toda se atualiza.

A mesma sequência de filtro em Python/pandas (independe do Streamlit):

In [ ]:
# Pipeline de filtro usado no app_04 (aqui, com valores fixos de exemplo)
estado = 'SP'
categorias = ['Informática', 'Eletronicos', 'Roupas']

filtro = df[df['estado'] == estado]
filtro = filtro[filtro['categoria'].isin(categorias)]

print('Registros apos o filtro (SP + categorias escolhidas):', len(filtro))

## 2.2 Layout: `app_05_sidebar_layout.py`

Containers para organizar a interface:

```python
st.sidebar.title('Filtros')                    # painel lateral
col1, col2 = st.columns([1, 2])                # larguras proporcionais
with col1:
    st.metric(...)
with st.expander('Ver detalhes'):              # conteudo recolhido
    st.dataframe(...)
aba1, aba2 = st.tabs(['Geral', 'Detalhe'])     # abas no corpo
with aba1:
    st.bar_chart(...)
```

Boa prática: sidebar para **filtros globais**, corpo para **resultados**.

## 2.3 Métricas, Formulários e Download: `app_06_metricas_form.py`

- `st.metric(label, valor, delta=None)` — indicador em destaque, com variação opcional.
- `st.progress(0.0–1.0)` — barra de progresso.
- `st.form('chave')` + `st.form_submit_button` — agrupa campos; o código do bloco só roda no clique do botão.
- `st.download_button(label, data, file_name, mime)` — faz download de um arquivo gerado (CSV, por exemplo).

### Sessão (importante!)

Como o script roda de cima a baixo a cada interação, variáveis **comuns são recriadas**. Para manter valores, use `st.session_state`:

```python
if 'cliques' not in st.session_state:
    st.session_state['cliques'] = 0

if st.button('Clique'):
    st.session_state['cliques'] += 1
```

In [ ]:
# As metricas do app_06 sao calculos simples de pandas:
receita = df['valor_venda'].sum()
pedidos = len(df)
ticket = df['valor_venda'].mean()

print(f'Receita: R$ {receita:,.0f}'.replace(',', '.'))
print(f'Pedidos: {pedidos}')
print(f'Ticket medio: R$ {ticket:,.2f}'.replace(',', '.'))

## 2.4 Matplotlib no App: `app_07_pyplot_matplotlib.py`

Todo gráfico do **Matplotlib** pode entrar na página com `st.pyplot(fig)`, reaproveitando o conteúdo do curso:

```python
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.barh(top.index, top['valor_venda'])
st.pyplot(fig)
plt.close(fig)
```

No topo do arquivo usamos `matplotlib.use('Agg')` (backend sem janela), próprio para apps web.

# MÓDULO 3 — Nível Avançado
## Cache e o dashboard completo

## 3.1 Cache: `app_08_cache.py`

Como o script roda **a cada interação**, tarefas pesadas (leitura de arquivo, consultas, agregações) se repetiriam toda vez. O decorador `st.cache_data` guarda o resultado por parâmetro:

```python
@st.cache_data
def carregar_dados():
    df = pd.read_csv('vendas.csv', sep=';')
    return df

@st.cache_data
def filtrar(regiao, categoria):
    ...  # so recalcula quando os argumentos mudam
```

Na segunda vez que o usuário pedir os **mesmos parâmetros**, o resultado vem do cache (memória/disco) sem repetir o cálculo. `st.cache_data.clear()` limpa tudo.

> Dica: `st.session_state` guarda **valores** entre reexecuções; `st.cache_data` acelera **funções**.

## 3.2 Dashboard Final: `app_09_dashboard.py`

Projeto que reúne todo o tutorial em um único painel:

- Sidebar com filtros combinados (região, categoria, status, faixa de valor, período).
- Linha de **métricas**: receita, pedidos, ticket médio, estados atendidos.
- Abas com **evolução mensal**, ranking por **categoria**, por **estado** e **dados detalhados**.
- Botão de **download** do recorte filtrado em CSV.

É o ponto de partida ideal para o projeto final do curso: troque o `vendas.csv` pelos dados reais e ajuste os rótulos.

## 3.3 Como Publicar um App

- **Local:** `streamlit run app_09_dashboard.py`
- **Nuvem (gratuita):** publique o repositório (GitHub) no **Streamlit Community Cloud** — sem configuração de servidor.
- Sempre versionar o `requirements.txt` (ex.: `streamlit`, `pandas`, `matplotlib`).

In [ ]:
# Verificacao final: todos os apps presentes e sintaticamente validos
import pathlib
import py_compile

apps = sorted(pathlib.Path('.').glob('app_*.py'))
for app in apps:
    py_compile.compile(str(app), doraise=True)

print(f'{len(apps)} apps compilados com sucesso:')
for app in apps:
    print(' -', app.name)

---
# Resumo Rápido de Referência

| Comando | O que faz |
|---------|-----------|
| `streamlit run app.py` | Roda o app no navegador |
| `st.title` / `st.header` / `st.subheader` / `st.caption` | Hierarquia de títulos |
| `st.write(obj)` | Elemento universal (texto, DF, dict, figura) |
| `st.dataframe(df)` / `st.table(df)` | Tabela interativa / estática |
| `st.line_chart` / `st.bar_chart` / `st.area_chart` | Gráficos nativos |
| `st.pyplot(fig)` | Figura do Matplotlib na página |
| `st.sidebar` | Painel lateral de controles |
| `st.columns` / `st.expander` / `st.tabs` | Containers de layout |
| `st.selectbox` / `st.multiselect` / `st.slider` / `st.radio` | Widgets de filtro |
| `st.metric(label, valor, delta)` | Indicador em destaque |
| `st.progress(0.0–1.0)` | Barra de progresso |
| `st.form` + `st.form_submit_button` | Agrupa campos por formulário |
| `st.download_button(...)` | Download de arquivo gerado |
| `st.session_state['chave']` | Valor persistente entre reexecuções |
| `st.cache_data` / `st.cache_data.clear()` | Cache de funções pesadas |